In [2]:
import pandas as pd
df = pd.read_csv('data.csv')

In [15]:
df.describe()
df.head(2)

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad


In [4]:
# Basic Eda
# df.columns
# df.shape
# df['Risk'].value_counts() this shows how the target variable is distributed
df.isnull().sum()
# df.dtypes

Unnamed: 0            0
Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     183
Checking account    394
Credit amount         0
Duration              0
Purpose               0
Risk                  0
dtype: int64

There are several NULL values in Savings Account but we cannot fill them blindly, it is possible that the applicant does not have a savings account <br><br> Also, the **Job** column is a numerical value but it could be some job categories represented as numbers AND we **cannot let the model think 3>2>1>0 because that is wrong**

Now we will check what all there is in the categorical columns

In [11]:
for col in df.select_dtypes(include="object").columns:
    print(df[col].value_counts())

Sex
male      690
female    310
Name: count, dtype: int64
Housing
own     713
rent    179
free    108
Name: count, dtype: int64
Saving accounts
little        603
moderate      103
quite rich     63
rich           48
Name: count, dtype: int64
Checking account
little      274
moderate    269
rich         63
Name: count, dtype: int64
Purpose
car                    337
radio/TV               280
furniture/equipment    181
business                97
education               59
repairs                 22
domestic appliances     12
vacation/others         12
Name: count, dtype: int64
Risk
good    700
bad     300
Name: count, dtype: int64


**Now, under savings account and checking account there are different values like rich and moderate so we could add "no account" for NaN which could beuseful** <br> 

<hr> <br> 

### Individual Columns relation with target RISK <br>

grouping the dataset by values of Risk <br>
then checking the mean value of Credit amount for each Risk Group <br>
Basically, what is the average value of Credit Amount for people with Good and Bad Risk.<br>
This is a good way to see if the risk is correlated with the credit amount.<br>
And with the result, we have a small inference that defaulters tend to take larger credit amount loans.

In [17]:
df.groupby("Risk")["Credit amount"].mean()

Risk
bad     3938.126667
good    2985.457143
Name: Credit amount, dtype: float64

In [22]:
df.groupby("Risk")["Age"].mean() # NOT VERY USEFUL...

Risk
bad     33.963333
good    36.224286
Name: Age, dtype: float64

In [32]:
df.groupby("Risk")["Duration"].mean() # NOT VERY USEFUL...

Risk
bad     24.860000
good    19.207143
Name: Duration, dtype: float64

Now, how to check how categorical columns help finding the target? <br>
Rather than finding the mean after grouping the Risk, we find out what % is Good and Bad for each categorical value of the column Housing/Purpose/etc. <br>
This is done via Crosstab : it is for getting a cross table between 2 categorical columns for every combination of their values. <br>
In crosstab, pass the rows first then columns. <br>
Normalisation show percentages **AND** margins = True shows an additional ALL column to show distribution <br>



In [37]:
pd.crosstab(df["Housing"], df["Risk"], normalize="index", margins = True)
# shows the ones who live free have a bad risk


Risk,bad,good
Housing,,
free,0.407407,0.592593
own,0.260870,0.739130
rent,0.391061,0.608939
All,0.300000,0.700000


In [44]:
pd.crosstab(df["Purpose"], df["Risk"], values = df["Credit amount"], aggfunc="mean")
# this show the average credit amount when each of the Purpose is Bad or Good

pd.crosstab(df["Purpose"], df["Risk"], normalize="index")
# vacation purpose is more likely to be bad risk

Risk,bad,good
Purpose,,
business,0.350515,0.649485
car,0.314540,0.685460
domestic appliances,0.333333,0.666667
education,0.389831,0.610169
furniture/equipment,0.320442,0.679558
radio/TV,0.221429,0.778571
repairs,0.363636,0.636364
vacation/others,0.416667,0.583333
